# Active Appeals CCD MVP Payment Pending (Silver Layer)
<table style='float:left;'>
   <tbody>
      <tr>
         <td style='text-align: left;'><b>Name: </b></td>
         <td>ARIADM_CCD_PaymentPending</td>
      </tr>
      <tr>
         <td style='text-align: left;'><b>Description: </b></td>
         <td>Notebook responsible for the silver layer, common for all active appeal states.</td>
      </tr>
      <tr>
         <td style='text-align: left;'><b>First Created: </b></td>
         <td>May-2025</td>
      </tr>
      <tr>
         <th style='text-align: left;'><b>Changelog (JIRA ref/initials/date):</b></th>
         <th>Comments</th>
      </tr>
      <tr>
         <td style='text-align: left;'><a href="https://tools.hmcts.net/jira/browse/ARIADM-667">ARIADM-667</a>/NSA/SEP-2024</td>
         <td>Create Silver Staging tables: stg_paymentpending_combined, stg_create_paymentpending_json_content, stg_paymentpending_with_json_status</td>
      </tr>
  
   </tbody>
</table>

### Import packages

In [0]:
import dlt
import json
# from pyspark.sql.functions import when, col,coalesce, current_timestamp, lit, date_format
from pyspark.sql.functions import *

In [0]:
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()

print(f"env_code: {lz_key}")  # This won't be redacted
print(f"env_name: {env_name}")  # This won't be redacted

KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
print(f"KeyVault_name: {KeyVault_name}") 

In [0]:
# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")

# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"


# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")


# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

## Set Variables

In [0]:
# read_hive = False

# Setting variables for use in subsequent cells
bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS"
gold_outputs = "ARIADM/CCD/APPEALS"
# hive_schema = "ariadm_ccd_apl"
# key_vault = "ingest00-keyvault-sbox"
AppealState = "paymentPending"


# Print all variables
variables = {
    # "read_hive": read_hive,
    
    "bronze_path": bronze_path,
    "silver_path": silver_path,
    "gold_path": gold_path,
    # "html_path": html_path,
    "gold_outputs": gold_outputs,
    # "hive_schema": hive_schema,
    "key_vault": KeyVault_name,
    "AppealState": AppealState

}

display(variables)

## PaymentPending: Silver DLT staging table for gold transformation

## Transformation Functions

In [0]:
from pyspark.sql.functions import col, from_json, trim, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# Define the schema of the JSON structure in caseManagementCategory
caseManagementCategory_json_schema = StructType([
    StructField("value", StructType([
        StructField("code", StringType(), True),
        StructField("label", StringType(), True)
    ]), True),
    StructField("list_items", ArrayType(
        StructType([
            StructField("code", StringType(), True),
            StructField("label", StringType(), True)
        ])
    ), True)
])

In [0]:

# AppealType grouping
def AppealType(silver_m1):
    df = silver_m1.select(
        col("CaseNo"),
        when(
            ((col("representation").isin('LR', 'AIP')) & (col("appealType").isNotNull())),
            coalesce(col("appealType"), lit(""))
        ).otherwise("").alias("appealType"),
        when(
            ((col("representation").isin('LR', 'AIP')) & (col("appealType").isNotNull())),
            coalesce(col("hmctsCaseCategory"), lit(""))
        ).otherwise("").alias("hmctsCaseCategory"),
        col("CaseNo").alias("appealReferenceNumber"),
        when(
            ((col("representation").isin('LR', 'AIP')) & (col("appealType").isNotNull())),
            coalesce(col("appealTypeDescription"), lit(""))
        ).otherwise("").alias("appealTypeDescription"),
        when(
            ((col("representation").isin('LR', 'AIP')) & (col("appealType").isNotNull())),
            from_json(col("caseManagementCategory"), caseManagementCategory_json_schema)
        ).otherwise(lit(None)).alias("caseManagementCategory"),
        lit("YES").alias("isAppealReferenceNumberAvailable"),
        lit("").alias("ccdReferenceNumberForDisplay")
    )

    # df = df.groupBy(col("CaseNo")).agg(
    #     collect_list(
    #         struct(
    #             'appealType', 'appealReferenceNumber', 'hmctsCaseCategory', 'appealTypeDescription', 'caseManagementCategory', 'isAppealReferenceNumberAvailable','ccdReferenceNumberForDisplay'
    #         )
    #     ).alias("appealType")
    # )
    return df

In [0]:
from pyspark.sql.functions import collect_list, struct, when, lit, col, max as spark_max

# caseData grouping
def caseData(silver_m5, silver_m1, silver_m3):
    silver_m1 = silver_m1.filter( ((col("representation").isin('LR', 'AIP')) & (col("appealType").isNotNull())))


    ##################################################
    # try window funtion instead of self join
    ###################################################
    # Filter silver_m3 to get rows with max StatusId and Outcome is not null
    silver_m3_grouped = silver_m3.groupBy("CaseNo").agg(
        spark_max("StatusId").alias("MaxStatusId")
    )
    
    silver_m3_filtered = silver_m3_grouped.alias("sm2").join(
        silver_m3.alias("sm3"),
        (col("sm3.CaseNo") == col("sm2.CaseNo")) & (col("sm3.StatusId") == col("sm2.MaxStatusId")),
        "inner"
    ).filter(col("sm3.Outcome").isNotNull()).select(col("sm3.CaseNo"), lit("Yes").alias("recordedOutOfTimeDecision"))

    ##############################################

    df = silver_m5.groupBy("CaseNo").agg(
        when(collect_list("LinkNo").isNotNull(), collect_list("LinkNo")).otherwise(lit([])).alias("caseLinks"),
        lit("No").alias("hasOtherAppeals")
    ).join(
        silver_m1.select("CaseNo", when(col("OutOfTimeIssue") == 1, "Yes").otherwise("No").alias("submissionOutOfTime")),
        on="CaseNo",
        how="inner"
    ).join(
        silver_m3_filtered,
        on="CaseNo",
        how="left"
    )
    
    return df

    

# AppealState = "paymentPending"
# silver_m5 = spark.table("ariadm_active_appeals.silver_link_detail").filter(col("TargetState") == lit(AppealState))
# silver_m1 = spark.table("ariadm_active_appeals.silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()
# silver_m3 = spark.table("ariadm_active_appeals.silver_status_detail").filter(col("TargetState") == lit(AppealState))
# df = caseData(silver_m5, silver_m1, silver_m3)
# display(df)

In [0]:
def main_paymentPending():

    AppealState = "paymentPending"

    ## read all silver tables
    silver_m1 = spark.table("ariadm_active_appeals.silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()
    # silver_m2 = spark.table("ariadm_active_appeals.silver_caseapplicant_detail").filter(col("TargetState") == lit(AppealState))
    silver_m3 = spark.table("ariadm_active_appeals.silver_status_detail").filter(col("TargetState") == lit(AppealState))
    # silver_m4 = spark.table("ariadm_active_appeals.silver_transaction_detail").filter(col("TargetState") == lit(AppealState))
    silver_m5 = spark.table("ariadm_active_appeals.silver_link_detail").filter(col("TargetState") == lit(AppealState))
    # silver_m6 = spark.table("ariadm_active_appeals.silver_adjudicator_detail").filter(col("TargetState") == lit(AppealState))
    # silver_m7 = spark.table("ariadm_active_appeals.silver_appealcategory_detail").filter(col("TargetState") == lit(AppealState))
    # silver_m8 = spark.table("ariadm_active_appeals.silver_documentsreceived_detail").filter(col("TargetState") == lit(AppealState))
    # silver_m9 = spark.table("ariadm_active_appeals.silver_history_detail").filter(col("TargetState") == lit(AppealState))

    # Aggregate details
    AppealType_df = AppealType(silver_m1)
    # grouped_transaction = TransactionDetails(silver_m4)
    caseData_df = caseData(silver_m5, silver_m1, silver_m3)

    # Join all aggregated data with Appeal Case Details
    df_combined = AppealType_df.join(caseData_df, on="CaseNo", how="left")

    # # Extract the first element of the array appealType[0]
    # df_exploded = df_combined.withColumn("appealType_first", col("appealType")[0])

    # # Flatten the struct inside appealType_first
    # flattened_df = df_exploded.select(
    #     col("CaseNo"),
    #     *[col("appealType_first")[field.name].alias(field.name) for field in df_exploded.schema["appealType_first"].dataType.fields]
    # )

    # Create JSON and filename
    df_final = df_combined.withColumn(
        "JSON_Content", to_json(struct(*df_combined.drop(col("CaseNo")).columns))
    ).withColumn(
        "JSON_File_name", concat(lit(f"{gold_outputs}/JSON/APPEALS_"), regexp_replace(col("CaseNo"), "/", "_"), lit(".json"))
    )

    return df_final

In [0]:

# AppealState = "paymentPending"
# silver_m5 = spark.table("ariadm_active_appeals.silver_link_detail").filter(col("TargetState") == lit(AppealState))
# silver_m1 = spark.table("ariadm_active_appeals.silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()
# silver_m3 = spark.table("ariadm_active_appeals.silver_status_detail").filter(col("TargetState") == lit(AppealState))
# df = caseData(silver_m5, silver_m1, silver_m3)
# display(df)

In [0]:
# AppealState = "paymentPending"
# silver_m1 = spark.table("ariadm_active_appeals.silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()
# df = AppealType(silver_m1)
# display(df)

In [0]:
# df.printSchema()

In [0]:
# df_final = main_paymentPending()
# display(df_final.select("*"))

In [0]:
# df_final.printSchema()

In [0]:
secret = dbutils.secrets.get(KeyVault_name, "CURATED-sbox-SAS-TOKEN")

In [0]:
from azure.storage.blob import BlobServiceClient, BlobClient, ContainerClient
import os

# Set up the BlobServiceClient with your connection string
connection_string = secret

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Specify the container name
container_name = "gold"
container_client = blob_service_client.get_container_client(container_name)

In [0]:
# Upload HTML to Azure Blob Storage
def upload_to_blob(file_name, file_content):
    try:
        # blob_client = container_client.get_blob_client(f"{gold_outputs}/HTML/{file_name}")
        blob_client = container_client.get_blob_client(f"{file_name}")
        blob_client.upload_blob(file_content, overwrite=True)
        return "success"
    except Exception as e:
        return f"error: {str(e)}"

# Register the upload function as a UDF
upload_udf = udf(upload_to_blob)



# df_with_upload_status = df_final.withColumn(
#     "Status", upload_udf(col("JSON_File_name"), col("JSON_Content"))
# )

# display(df_with_upload_status)


In [0]:
import dlt
from pyspark.sql.functions import col, lit

@dlt.table(
    name="stg_apl_create_json_content",
    comment="Delta Live unified stage Gold Table for gold outputs.",
    path=f"{silver_path}/stg_apl_create_json_content"
)
@dlt.expect_or_fail("valid_caseno_exists", "valid_CaseNo IS NOT NULL")
@dlt.expect_or_fail("valid_appealtype_exists", "valid_AppealType IS NOT NULL OR AppealType == ''")
def stg_apl_create_json_content():
    try:
        silver_m1 = dlt.read("silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()
        bronze_appealtype_lookup_df = dlt.read("bronze_appealtype").select(col("AppealType")).distinct()
        # silver_m2 = dlt.read("silver_caseapplicant_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m3 = dlt.read("silver_status_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m4 = dlt.read("silver_transaction_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m5 = dlt.read("silver_link_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m6 = dlt.read("silver_adjudicator_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m7 = dlt.read("silver_appealcategory_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m8 = dlt.read("silver_documentsreceived_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m9 = dlt.read("silver_history_detail").filter(col("TargetState") == lit(AppealState))
    except:
        silver_m1 = spark.table("ariadm_active_appeals.silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()
        bronze_appealtype_lookup_df = spark.table("ariadm_active_appeals.bronze_appealtype").select(col("AppealType")).distinct()
        # silver_m2 = spark.table("ariadm_active_appeals.silver_caseapplicant_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m3 = spark.table("ariadm_active_appeals.silver_status_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m4 = spark.table("ariadm_active_appeals.silver_transaction_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m5 = spark.table("ariadm_active_appeals.silver_link_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m6 = spark.table("ariadm_active_appeals.silver_adjudicator_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m7 = spark.table("ariadm_active_appeals.silver_appealcategory_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m8 = spark.table("ariadm_active_appeals.silver_documentsreceived_detail").filter(col("TargetState") == lit(AppealState))
        # silver_m9 = spark.table("ariadm_active_appeals.silver_history_detail").filter(col("TargetState") == lit(AppealState))

    df_final = main_paymentPending()

    # Join with valid CaseNo reference
    valid_case_df = silver_m1.select("CaseNo").distinct()
    df_joined = df_final.join(valid_case_df.withColumnRenamed("CaseNo", "valid_CaseNo"),
                              df_final["CaseNo"] == col("valid_CaseNo"),
                              "left")

    # Add expectation for valid CaseNo
    # dlt.expect_or_fail("valid_caseno_exists", col("valid_CaseNo").isNotNull())

    # Join with valid AppealType reference
    df_joined = df_joined.join(
        bronze_appealtype_lookup_df.withColumnRenamed("AppealType", "valid_AppealType"),
        df_joined["AppealType"] == col("valid_AppealType"),
        "left"
    )

    # Add expectation for valid AppealType
    # dlt.expect_or_fail("valid_appealtype_exists", col("valid_AppealType").isNotNull())

    return df_joined.drop("valid_CaseNo", "valid_AppealType")


## Gold Outputs and Tracking DLT Table Creation

In [0]:
@dlt.table(
    name="gold_appeals_with_json",
    comment="Delta Live Gold Table with JSON content.",
    path=f"{silver_path}/gold_appeals_with_json"
)
def gold_appeals_with_json():
    """
    Delta Live Table for creating and uploading JSON content for Appeals.
    """
    # Load source data
    df_unified = dlt.read("stg_apl_create_json_content")
    

    # Optionally load data from Hive if needed
    # if read_hive:
    #     df_unified = spark.read.table(f"hive_metastore.{hive_schema}.stg_appeals_unified")

    # Repartition to optimize parallelism
    repartitioned_df = df_unified.repartition(64)

    df_with_upload_status = repartitioned_df.filter(~col("JSON_content").like("Error%")).withColumn(
            "Status", upload_udf(col("JSON_File_Name"), col("JSON_content"))
        )


    # Return the DataFrame for DLT table creation
    return df_with_upload_status.select("CaseNo", "JSON_content",col("JSON_File_Name").alias("File_Name"),"Status")


In [0]:
dbutils.notebook.exit("Notebook completed successfully")

## Appendix

In [0]:
# try:
#     silver_m1 = dlt.read("silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()
#     bronze_appealtype_df = dlt.read("bronze_appealtype").select(col("AppealType")).distinct()
#     # silver_m2 = dlt.read("silver_caseapplicant_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m3 = dlt.read("silver_status_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m4 = dlt.read("silver_transaction_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m5 = dlt.read("silver_link_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m6 = dlt.read("silver_adjudicator_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m7 = dlt.read("silver_appealcategory_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m8 = dlt.read("silver_documentsreceived_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m9 = dlt.read("silver_history_detail").filter(col("TargetState") == lit(AppealState))
# except:
#     silver_m1 = spark.table("ariadm_active_appeals.silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()
#     bronze_appealtype_df = spark.table("ariadm_active_appeals.bronze_appealtype").select(col("AppealType")).distinct()
#     # silver_m2 = spark.table("ariadm_active_appeals.silver_caseapplicant_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m3 = spark.table("ariadm_active_appeals.silver_status_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m4 = spark.table("ariadm_active_appeals.silver_transaction_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m5 = spark.table("ariadm_active_appeals.silver_link_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m6 = spark.table("ariadm_active_appeals.silver_adjudicator_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m7 = spark.table("ariadm_active_appeals.silver_appealcategory_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m8 = spark.table("ariadm_active_appeals.silver_documentsreceived_detail").filter(col("TargetState") == lit(AppealState))
#     # silver_m9 = spark.table("ariadm_active_appeals.silver_history_detail").filter(col("TargetState") == lit(AppealState))


# df_final = main_paymentPending()

# # Add expectation that fails the pipeline if CaseNo in df_final is not present in silver_m1
# valid_case_df = silver_m1.select("CaseNo").distinct()
# df_joined = df_final.join(valid_case_df.withColumnRenamed("CaseNo", "valid_CaseNo"),
#                             df_final["CaseNo"] == col("valid_CaseNo"),
#                             "left")

# df_with_expectation = dlt.expect_or_fail("valid_caseno_exists", col("valid_CaseNo").isNotNull())

### Validation

In [0]:
# ## Validation Framework

# from pyspark.sql.types import *

# def validate_AppealType():
  
#   test_df = AppealType(silver_m1)

#   expected_types = {
#     "appealReferenceNumber": StringType(),
#     "isAppealReferenceNumberAvailable": Stringtype()

#   }
      
#   nested_types = test_df.schema["appealType"].dataType

#   for field in nested_types.elementType.fields:
#     field_name = field.name
#     input_type = field.dataType
#     # print(f"Field: {field_name}, incoming type {input_type}")

#     expected_type = expected_types[field_name]

#     assert input_type == expected_type, f"Expected type {expected_type} but got {input_type}"
#     print(f"Succesfully validated type {expected_type} for field {field_name}")


In [0]:
# from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType, IntegerType




In [0]:
# from datetime import datetime

# def validate_AppealType_content():
#     schema = StructType([
#         StructField("CaseNo", StringType(), True),
#         StructField("CasePrefix", StringType(), True),
#         StructField("OutOfTimeIssue", BooleanType(), True),
#         StructField("DateLodged", TimestampType(), True),
#         StructField("DateAppealReceived", TimestampType(), True),
#         StructField("CentreId", IntegerType(), True),
#         StructField("NationalityId", IntegerType(), True),
#         StructField("AppealTypeId", IntegerType(), True),
#         StructField("DeportationDate", TimestampType(), True),
#         StructField("RemovalDate", TimestampType(), True),
#         StructField("VisitVisaType", IntegerType(), True),
#         StructField("DateOfApplicationDecision", TimestampType(), True),
#         StructField("HORef", StringType(), True),
#         StructField("InCamera", BooleanType(), True),
#         StructField("CourtPreference", IntegerType(), True),
#         StructField("LanguageId", IntegerType(), True),
#         StructField("Interpreter", IntegerType(), True),
#         StructField("RepresentativeId", IntegerType(), True),
#         StructField("CaseRepName", StringType(), True)
#     ])
    
#     data = [
#         ("12345", "CP", True, datetime(2025, 6, 24, 0, 0, 0), datetime(2025, 6, 24, 0, 0, 0), 1, 1, 1, datetime(2025, 6, 24, 0, 0, 0), datetime(2025, 6, 24, 0, 0, 0), 1, datetime(2025, 6, 24, 0, 0, 0), "HO123", True, 1, 1, 1, 1, "RepName")
#     ]
    
#     df = spark.createDataFrame(data, schema)

#     appealtype_df = AppealType(df)
#     appealtype_df.display()

#     ### Some assert Content Test e.g appealReferenceNumber is in format first 3 characters are letters followed by/ followed by 3 numbers etc

# validate_AppealType_content()

In [0]:
# import json

# first_row = df_final.select("JSON_Content").first()
# json_content = first_row["JSON_Content"]
# parsed_json = json.loads(json_content)

# def validate_json_types(parsed_json):
#     expected_types = {
#         "appealReferenceNumber": str,
#         "appealType": list,
#         "Transactions": list
#     }
    
#     inner_expected_types = {
#         "Transactions": {
#             "transactionId": str,
#             "amount": float,
#             "date": str
#         },
#         "appealType": {
#             "appealReferenceNumber": str,
#             "isAppealReferenceNumberAvailable": str
#         }
#     }
    
#     for key, expected_type in expected_types.items():
#         if key in parsed_json:
#             if not isinstance(parsed_json[key], expected_type):
#                 print(f"Validation failed: Key '{key}' has incorrect type. Expected {expected_type}, got {type(parsed_json[key])}")
#                 raise TypeError(f"Key '{key}' has incorrect type. Expected {expected_type}, got {type(parsed_json[key])}")
#             else:
#                 print(f"Validation passed: Key '{key}' has correct type {expected_type}")
#                 if key in inner_expected_types:
#                     for item in parsed_json[key]:
#                         for inner_key, inner_expected_type in inner_expected_types[key].items():
#                             if inner_key in item:
#                                 if not isinstance(item[inner_key], inner_expected_type):
#                                     print(f"Validation failed: Key '{inner_key}' in '{key}' has incorrect type. Expected {inner_expected_type}, got {type(item[inner_key])}")
#                                     raise TypeError(f"Key '{inner_key}' in '{key}' has incorrect type. Expected {inner_expected_type}, got {type(item[inner_key])}")
#                                 else:
#                                     print(f"Validation passed: Key '{inner_key}' in '{key}' has correct type {inner_expected_type}")
#                                 if inner_key == "isAppealReferenceNumberAvailable" and item[inner_key] not in ["YES", "no"]:
#                                     print(f"Validation failed: Key '{inner_key}' in '{key}' has invalid value. Expected 'YES' or 'no', got {item[inner_key]}")
#                                     raise ValueError(f"Key '{inner_key}' in '{key}' has invalid value. Expected 'YES' or 'no', got {item[inner_key]}")

# validate_json_types(parsed_json)
# print(json.dumps(parsed_json, indent=4))

### Analysis

In [0]:
df_final = main_paymentPending()
display(df_final.select("*"))

In [0]:
import json

first_row = df_final.filter(df_final["CaseNo"] == "HU/00035/2017").select("JSON_Content").first()
json_content = first_row["JSON_Content"]
parsed_json = json.loads(json_content)
display(parsed_json)

In [0]:
# from pyspark.sql.functions import col, lit, when, coalesce, collect_list, struct

# AppealState = "paymentPending"  # Define AppealState variable

# silver_m1 = spark.table("ariadm_active_appeals.silver_appealcase_detail").filter(col("TargetState") == lit(AppealState)).distinct()

# df = silver_m1.select(
#     col("CaseNo"),
#     when(
#         (col("representation") == 'LR') | (col("representation") == 'AIP'),
#         coalesce(col("appealType"), lit(""))
#     ).otherwise("").alias("appealType"),
#     when(
#         (col("representation") == 'LR') | (col("representation") == 'AIP'),
#         coalesce(col("hmctsCaseCategory"), lit(""))
#     ).otherwise("").alias("hmctsCaseCategory"),
#     col("CaseNo").alias("appealReferenceNumber"),
#     when(
#         (col("representation") == 'LR') | (col("representation") == 'AIP'),
#         coalesce(col("appealTypeDescription"), lit(""))
#     ).otherwise("").alias("appealTypeDescription"),
#     when(
#         (col("representation") == 'LR'),
#         col("caseManagementCategory")
#     ).otherwise(lit(None)).alias("caseManagementCategory"),
#     lit("YES").alias("isAppealReferenceNumberAvailable"),
#     lit("").alias("ccdReferenceNumberForDisplay")
# )

# df = df.groupBy(col("CaseNo")).agg(
#     collect_list(
#         struct(
#             'appealType', 'appealReferenceNumber', 'hmctsCaseCategory', 'appealTypeDescription', 'caseManagementCategory', 'isAppealReferenceNumberAvailable','ccdReferenceNumberForDisplay'
#         )
#     ).alias("appealType")
#     )

# # .withColumn("caseManagementCategory", 
# #     expr("""
# #     struct(
# #         struct(
# #             caseManagementCategory as code,
# #             caseManagementCategory as label
# #         ) as value,
# #         array(
# #             struct(
# #                 caseManagementCategory as code,
# #                 caseManagementCategory as label
# #             )
# #         ) as list_items
# #     )""")

# display(df)

In [0]:
# from pyspark.sql.functions import col, count

# # Reading tables into DataFrames and labeling as M1 to M9
# M1 = spark.table("ariadm_active_appeals.silver_appealcase_detail").distinct()
# M2 = spark.table("ariadm_active_appeals.silver_caseapplicant_detail")
# M3 = spark.table("ariadm_active_appeals.silver_status_detail")
# M4 = spark.table("ariadm_active_appeals.silver_transaction_detail")
# M5 = spark.table("ariadm_active_appeals.silver_link_detail")
# M6 = spark.table("ariadm_active_appeals.silver_adjudicator_detail")
# M7 = spark.table("ariadm_active_appeals.silver_appealcategory_detail")
# M8 = spark.table("ariadm_active_appeals.silver_documentsreceived_detail")
# M9 = spark.table("ariadm_active_appeals.silver_history_detail")

# # Function to check for duplicates
# def check_duplicates(df, table_name):
#     duplicates = df.groupBy("caseno").agg(count("*").alias("count")).filter(col("count") > 1)
#     if duplicates.count() > 0:
#         displayHTML(f"<span style='color:red;'>&#x274C; Table {table_name} has duplicates.</span>")
#     else:
#         displayHTML(f"<span style='color:green;'>&#x2705; Table {table_name} has no duplicates.</span>")

# # Check for duplicates in each table
# check_duplicates(M1, "silver_appealcase_detail")
# check_duplicates(M2, "silver_caseapplicant_detail")
# check_duplicates(M3, "silver_status_detail")
# check_duplicates(M4, "silver_transaction_detail")
# check_duplicates(M5, "silver_link_detail")
# check_duplicates(M6, "silver_adjudicator_detail")
# check_duplicates(M7, "silver_appealcategory_detail")
# check_duplicates(M8, "silver_documentsreceived_detail")
# check_duplicates(M9, "silver_history_detail")